In [ ]:
import os

# Run this cell once to save shared credentials to disk.
# AZURE_API_KEY is intentionally not saved here -- set it manually each session.
with open(os.path.expanduser('~/.env_coliseum'), 'w') as f:
    f.write('AZURE_BASE_URL=https://rsgd15-foundry.openai.azure.com/openai/v1/\n')
    f.write('NGROK_TOKEN=2gzfvECda2xLWu1rzx5HsV4DqdU_4yCynLfeFQSQDHua7Nymd\n')
print('Saved to ~/.env_coliseum')
print('Still needed: os.environ["AZURE_API_KEY"] = "your_key"')

In [ ]:
import sys, os, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "flask", "flask-cors", "pyngrok", "openai", "requests", "-q"])
for url in [
    "https://raw.githubusercontent.com/omar-florez/agent-coliseum/bcp/agent_base.py",
    "https://raw.githubusercontent.com/omar-florez/agent-coliseum/bcp/agent_server.py",
]:
    subprocess.check_call(["wget", "-q", "-N", url])
os.environ["AZURE_API_KEY"]  = "YOUR_AZURE_API_KEY_HERE"
os.environ["AZURE_BASE_URL"] = "https://rsgd15-foundry.openai.azure.com/openai/v1/"
os.environ["NGROK_TOKEN"]    = "YOUR_NGROK_TOKEN_HERE"
print(f"Python: {sys.executable}")
print("Setup complete")

In [ ]:
# ============================================================
#  AGENT COLISEUM — BCP — Colab 03: Naive Baseline
# ============================================================
#  Intentionally simple — no memory, no RAG, no strategy.
#  Used to contrast with Colabs 01 and 02 during the demo.
#
#  TALK NARRATIVE — why this agent loses:
#    No history awareness, no memory, no retrieved facts.
#    Same LLM, same Azure key, radically different performance.
# ============================================================

import os, random, re
from openai import OpenAI
from agent_base import Agent, MatchContext, MatchResult, WorldContext, Position
from agent_server import serve_and_register

# ── Config ────────────────────────────────────────────────────────────────
# Load shared credentials from ~/.env_coliseum
_env = os.path.expanduser('~/.env_coliseum')
if os.path.exists(_env):
    for _l in open(_env):
        _l = _l.strip()
        if _l and '=' in _l:
            _k, _v = _l.split('=', 1)
            os.environ[_k] = _v

AZURE_API_KEY  = os.environ.get('AZURE_API_KEY',  '')
AZURE_BASE_URL = os.environ.get('AZURE_BASE_URL', 'https://rsgd15-foundry.openai.azure.com/openai/v1/')
MODEL          = 'gpt-5'
ARENA_URL      = 'https://agent-coliseum.onrender.com'
NGROK_TOKEN    = os.environ.get('NGROK_TOKEN', '')

if not AZURE_API_KEY:
    print('WARNING: AZURE_API_KEY not set. Run: os.environ["AZURE_API_KEY"] = "your_key"')
else:
    print(f'AZURE_API_KEY: ...{AZURE_API_KEY[-6:]}')
    print(f'AZURE_BASE_URL: {AZURE_BASE_URL}')
    print(f'NGROK_TOKEN: ...{NGROK_TOKEN[-6:]}')

client = OpenAI(base_url=AZURE_BASE_URL, api_key=AZURE_API_KEY)
print(f"Client ready: {client.base_url}")

# ── Agent ─────────────────────────────────────────────────────────────────

class NaiveAgent(Agent):
    name        = "Bot Basico"
    avatar      = "🤖"
    description = "Un agente simple sin memoria ni recuperacion de conocimiento"

    def think(self, ctx: MatchContext) -> str:
        role_instruction = (
            f"Generate a sharp, specific question about {ctx.topic} to ask your opponent."
            if ctx.role == "asker" else
            f"Answer this question accurately in 1-2 sentences: {ctx.current_question}"
        )

        prompt = f"""You are competing in a Latin America knowledge tournament.
Topic: {ctx.topic} | Role: {ctx.role}

Task: {role_instruction}

Reason step by step. Write a concrete response under each label (do NOT leave blanks):

SITUATION [Wei et al. 2022 — Chain-of-Thought]: <state of match>
OPPONENT [Langner et al. 2023 — Theory of Mind]: <their weaknesses>
GOAL [Yao et al. 2022 — ReAct]: <your objective this turn>
DRAFT [Nye et al. 2021 — Scratchpad]: <first attempt>
CRITIQUE [Madaan et al. 2023 — Self-Refine]: <how to improve>
FINAL [Shinn et al. 2023 — Reflexion]: <your {'question' if ctx.role == 'asker' else 'answer'}, concise>"""

        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_completion_tokens=250,
        )
        return response.choices[0].message.content.strip()

    def ask(self, ctx: MatchContext) -> str:
        return self._extract_final(self.think(ctx))

    def answer(self, ctx: MatchContext) -> str:
        return self._extract_final(self.think(ctx))

    def _extract_final(self, text: str) -> str:
        """
        Extract the FINAL answer. Robust to GPT formatting variations.
        Handles: "FINAL: ...", "**FINAL:**", FINAL on its own line.
        Falls back to last non-empty non-comment line.
        """
        lines = text.split("\n")
        for i, line in enumerate(lines):
            clean = line.replace("**", "").strip()
            if clean.upper().startswith("FINAL"):
                # Remove "FINAL:" or "FINAL [...]:" prefix
                after = clean[5:].lstrip(":").strip()
                # Strip any [...] annotation
                import re
                after = re.sub(r"^\[.*?\]\s*:?\s*", "", after).strip()
                if after:
                    return after
                for j in range(i+1, len(lines)):
                    if lines[j].strip():
                        return lines[j].strip()
        for line in reversed(lines):
            if line.strip() and not line.strip().startswith("#"):
                return line.strip()
        return text.strip()

# ── Keepalive ─────────────────────────────────────────────────────────────
import threading, requests as _req, time as _time

def _keepalive(arena_url):
    """Ping arena every 60s to keep ngrok tunnel and Colab session alive."""
    while True:
        _time.sleep(60)
        try:
            _req.get(f"{arena_url}/health", timeout=5)
        except:
            pass

threading.Thread(target=_keepalive, args=(ARENA_URL,), daemon=True).start()
print("Keepalive thread started")

# ── Instantiate ───────────────────────────────────────────────────────────
agent = NaiveAgent()

In [ ]:
# ── DEBUG CELL — inspect raw GPT-5 output before running the arena ────────
from agent_base import MatchContext

test_ctx = MatchContext(
    match_id="test", topic="Latin American geography",
    turn=1, total_turns=3, role="asker",
    history=[], my_agent_id="me", opponent_agent_id="opp",
    opponent_name="Test Opponent", my_scores=[], opponent_scores=[],
)
print("=== RAW think() OUTPUT (asker) ===")
raw = agent.think(test_ctx)
print(raw)
print()
print("=== EXTRACTED FINAL ===")
print(repr(agent._extract_final(raw)))
print()

test_ctx2 = MatchContext(
    match_id="test", topic="Latin American geography",
    turn=1, total_turns=3, role="answerer",
    history=[], my_agent_id="me", opponent_agent_id="opp",
    opponent_name="Test Opponent", my_scores=[], opponent_scores=[],
    current_question="What is the longest river in South America?",
)
print("=== RAW think() OUTPUT (answerer) ===")
raw2 = agent.think(test_ctx2)
print(raw2)
print()
print("=== EXTRACTED FINAL ===")
print(repr(agent._extract_final(raw2)))

In [ ]:
# ── Run cell ─────────────────────────────────────────────────────────────
from pyngrok import ngrok
ngrok.kill()  # kill any existing tunnels before starting

serve_and_register(
    agent       = agent,
    arena_url   = ARENA_URL,
    port        = 5000,
    ngrok_token = NGROK_TOKEN,
)
# This cell blocks. Agent is now live and registered.
# Wait for the organizer to accept you in the admin panel.